# QGSW ResUNet Training on a Central Patch — Multi-step Variant

This notebook is a variant of `SSHforecastingQGSW_PyTorch.ipynb` with these changes:

1. **GPU pre-transfer of the SSH archive.** `ssh_daily` is moved to the training device once, eliminating per-pair host→device copies inside the training and validation loops.
2. **Multi-step rollout loss with a 1→`max_rollout_days` curriculum.** Each training pair is supervised on `k_today` daily snapshots from a single forward call to the QGSW solver. Validation reports the per-day MSE curve up to `max_rollout_days`.
3. **`float32` training** by default, `AdamW` optimizer, **all validation pairs** evaluated each epoch, and an **optional hybrid SSH + PV loss** controlled by `loss_pv_weight` (set >0 to enable).

The QGSW solver also benefits from the host-device sync removed from `modelliptic.pv2h` (see `_safe_divide`).

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys
import time

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import xarray as xr
import torch
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt


def find_project_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "QGNet_Pytorch").is_dir():
            return base
        if base.name == "QGNet_Pytorch":
            return base.parent
        if base.name == "notebooks" and base.parent.name == "QGNet_Pytorch":
            return base.parent.parent
    raise FileNotFoundError("Could not locate the QGNet_Pytorch folder")


PROJECT_ROOT = find_project_root()
TORCH_ROOT = PROJECT_ROOT / "QGNet_Pytorch"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from QGNet_Pytorch.qgsw import qgsw as qgsw_torch

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Experiment Configuration

For quick one-sample overfitting, set `debug_overfit_one_sample = True`. The default path uses real epochs: each epoch traverses the full training split once, with a seeded reshuffle before batching. The rollout horizon ramps from 1 to `max_rollout_days` across the schedule (step curriculum). Validation reports the per-day MSE curve up to `max_rollout_days`.

In [ ]:
training_patch_size = 128
train_days = 365
validation_fraction_of_remainder = 0.5

random_seed = 0
num_epochs = 10
samples_per_batch = 8  # Gradient-accumulation batch size over SSH pairs. Set to None for one full-epoch batch.
max_val_samples = None # None = use all validation pairs; positive int = subset size; 0 = skip validation.
validation_interval = 1

training_dt = 600
nn_cadence_seconds = 3600
training_lr = 1e-3
lr_drop_factor_per_k_step = 0.5  # Multiplied into LR each time k_today increments. 1.0 keeps LR constant.
training_dtype = torch.float32
training_snu = 0.0
loss_mask_mode = "interior"  # "interior" or "finite"
loss_pv_weight = 0.0  # 0.0 = SSH MSE only; >0 enables hybrid SSH + loss_pv_weight * normalized PV MSE.

# Channels fed to the correction network, stacked in this order.
# ("ssh",) matches the ConvNet of arXiv:1911.08856, which takes SSH as input.
# Add "mask" if the patch contains land; "f0"/"dx"/"dy" are constant on a
# single fixed domain and then act only as a positional encoding.
training_nn_channels = ("ssh",)

resunet_base_channels = 16
resunet_depth = 2  # Number of pooling stages: 2 = original (bottleneck 32x32 for 128x128); 3 = bottleneck 16x16; >=4 typically too aggressive.
debug_overfit_one_sample = False
overfit_pair_offset = 0
max_rollout_days = 5  # Curriculum ramps 1 -> max_rollout_days; validation reports per-day MSE up to this horizon.

if debug_overfit_one_sample:
    num_epochs = max(num_epochs, 120)
    samples_per_batch = 1
    max_val_samples = 0
    validation_interval = None
    training_dt = 3600
    nn_cadence_seconds = 3600
    training_lr = 1e-3
    max_rollout_days = 1

training_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(training_dtype)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)


def cadence_steps_for_dt(dt_seconds, cadence_seconds):
    steps = int(round(float(cadence_seconds) / float(dt_seconds)))
    if steps < 1:
        raise ValueError("nn cadence must be at least one model time step")
    if not np.isclose(steps * float(dt_seconds), float(cadence_seconds)):
        raise ValueError("nn_cadence_seconds must be an integer multiple of training_dt")
    return steps


training_nn_cadence_steps = cadence_steps_for_dt(training_dt, nn_cadence_seconds)

print(f"Device: {training_device}")
print(f"Random seed: {random_seed}")
print(f"Training dtype: {training_dtype}")
print(f"Torch default dtype: {torch.get_default_dtype()}")
print(f"dt={training_dt} s, correction cadence={nn_cadence_seconds} s ({training_nn_cadence_steps} model steps)")
print(f"num_epochs={num_epochs}, samples_per_batch={samples_per_batch}, max_val_samples={max_val_samples}")
print(f"max_rollout_days={max_rollout_days}")

## 3. Load and Crop the Daily SSH Archive

The central patch is selected before any split or tensor transfer. If you provide `ssh_daily_np`, `ssh_daily_lon`, and `ssh_daily_lat` manually, the same crop is applied unless the arrays are already `128 x 128`.

In [ ]:
def find_osmo_sshdegrad_file():
    candidate_patterns = [
        "*osmosis*.nc",
        "*OSMOSIS*.nc",
        "*OSMO*.nc",
        "*SSHdegrad*.nc",
    ]
    for root in [TORCH_ROOT, PROJECT_ROOT]:
        for pattern in candidate_patterns:
            matches = sorted(root.glob(pattern))
            if matches:
                return matches[0]
    return None


def load_osmo_sshdegrad_dataset(path):
    with xr.open_dataset(path) as ds:
        if "degraded_sossheig" not in ds:
            raise KeyError(f"{path} does not contain variable degraded_sossheig")

        lon_name = "nav_lon" if "nav_lon" in ds else "lon"
        lat_name = "nav_lat" if "nav_lat" in ds else "lat"
        if lon_name not in ds or lat_name not in ds:
            raise KeyError(f"{path} must contain nav_lon/nav_lat or lon/lat")

        ssh = ds["degraded_sossheig"].where(ds["degraded_sossheig"] != 99999.0).values.astype(np.float64)
        lon_data = ds[lon_name].values.astype(np.float64)
        lat_data = ds[lat_name].values.astype(np.float64)
        time_data = ds["time_counter"].values if "time_counter" in ds else None

    if ssh.ndim != 3:
        raise ValueError(f"degraded_sossheig must have shape (nt, ny, nx), got {ssh.shape}")
    if lon_data.shape != ssh.shape[1:] or lat_data.shape != ssh.shape[1:]:
        raise ValueError(
            "Training lon/lat shapes must match degraded_sossheig spatial shape: "
            f"ssh={ssh.shape[1:]}, lon={lon_data.shape}, lat={lat_data.shape}"
        )
    return ssh, lon_data, lat_data, time_data


def central_patch_slices(spatial_shape, patch_size):
    ny, nx = map(int, spatial_shape)
    patch_size = int(patch_size)
    if patch_size <= 0:
        raise ValueError("training_patch_size must be positive")
    if ny < patch_size or nx < patch_size:
        raise ValueError(f"Cannot extract a {patch_size}x{patch_size} patch from spatial shape {(ny, nx)}")
    y0 = (ny - patch_size) // 2
    x0 = (nx - patch_size) // 2
    return slice(y0, y0 + patch_size), slice(x0, x0 + patch_size)


def crop_to_training_patch(ssh, lon_data, lat_data, c_data=None):
    ssh = np.asarray(ssh, dtype=np.float64)
    lon_data = np.asarray(lon_data, dtype=np.float64)
    lat_data = np.asarray(lat_data, dtype=np.float64)

    if ssh.ndim != 3:
        raise ValueError(f"ssh_daily_np must have shape (nt, ny, nx), got {ssh.shape}")
    if lon_data.shape != ssh.shape[1:] or lat_data.shape != ssh.shape[1:]:
        raise ValueError(
            "ssh_daily_lon and ssh_daily_lat must match ssh_daily_np spatial shape before cropping: "
            f"ssh={ssh.shape[1:]}, lon={lon_data.shape}, lat={lat_data.shape}"
        )

    source_shape = ssh.shape[1:]
    patch_shape = (int(training_patch_size), int(training_patch_size))
    if source_shape == patch_shape:
        y_slice, x_slice = slice(0, patch_shape[0]), slice(0, patch_shape[1])
    else:
        y_slice, x_slice = central_patch_slices(source_shape, training_patch_size)

    c_patch = None
    if c_data is not None:
        c_data = np.asarray(c_data, dtype=np.float64)
        if c_data.shape == source_shape:
            c_patch = c_data[y_slice, x_slice].copy()
        elif c_data.shape == patch_shape:
            c_patch = c_data.copy()
        else:
            raise ValueError(f"ssh_daily_c has shape {c_data.shape}; expected {source_shape} or {patch_shape}")

    patch_info = {
        "source_shape": source_shape,
        "patch_shape": patch_shape,
        "y_start": int(y_slice.start),
        "y_stop": int(y_slice.stop),
        "x_start": int(x_slice.start),
        "x_stop": int(x_slice.stop),
    }
    return ssh[:, y_slice, x_slice].copy(), lon_data[y_slice, x_slice].copy(), lat_data[y_slice, x_slice].copy(), c_patch, patch_info


if "ssh_daily_np" not in globals():
    osmo_sshdegrad_path = find_osmo_sshdegrad_file()
    if osmo_sshdegrad_path is None:
        raise FileNotFoundError("Define ssh_daily_np/lon/lat manually, or place an OSMO/SSHdegrad NetCDF file in QGNet_Pytorch")
    ssh_daily_np, ssh_daily_lon, ssh_daily_lat, ssh_daily_time = load_osmo_sshdegrad_dataset(osmo_sshdegrad_path)
    print(f"Loaded dataset: {osmo_sshdegrad_path}")
else:
    if "ssh_daily_lon" not in globals() or "ssh_daily_lat" not in globals():
        raise ValueError("When providing ssh_daily_np manually, also define ssh_daily_lon and ssh_daily_lat")
    if "ssh_daily_time" not in globals():
        ssh_daily_time = None

ssh_daily_c_input = globals().get("ssh_daily_c", None)
ssh_daily_np, ssh_daily_lon, ssh_daily_lat, ssh_daily_c, ssh_daily_patch_info = crop_to_training_patch(
    ssh_daily_np,
    ssh_daily_lon,
    ssh_daily_lat,
    ssh_daily_c_input,
)
if ssh_daily_c is None:
    ssh_daily_c = np.zeros_like(ssh_daily_lon, dtype=np.float64) + 1.5

print(f"Patch source shape: {ssh_daily_patch_info['source_shape']}")
print(
    "Patch slices: "
    f"y={ssh_daily_patch_info['y_start']}:{ssh_daily_patch_info['y_stop']}, "
    f"x={ssh_daily_patch_info['x_start']}:{ssh_daily_patch_info['x_stop']}"
)
print(f"Working SSH shape: {ssh_daily_np.shape}")
print(f"Finite SSH values: {np.isfinite(ssh_daily_np).sum()} / {ssh_daily_np.size}")

## 4. Build Train, Validation, and Test Splits

In [ ]:
if ssh_daily_np.shape[1:] != (training_patch_size, training_patch_size):
    raise ValueError(f"Expected a {(training_patch_size, training_patch_size)} patch, got {ssh_daily_np.shape[1:]}")

n_days_available = int(ssh_daily_np.shape[0])
n_pairs = n_days_available - 1
if n_pairs < 4:
    raise ValueError("ssh_daily_np must contain at least five daily fields")

# Reserve max_rollout_days at the tail of every split so multi-step rollouts have all targets in-split.
rollout_buffer = max(1, int(max_rollout_days))

train_day_count = min(train_days, n_days_available - 3)
train_day_count = max(2, train_day_count)
train_pair_indices = np.arange(0, max(0, train_day_count - rollout_buffer), dtype=int)
if train_pair_indices.size < 1:
    raise ValueError(
        f"No training pairs left after reserving {rollout_buffer} day(s) at the tail of the training split. "
        "Increase train_days or reduce max_rollout_days."
    )

remaining_start = train_day_count
remaining_end = max(remaining_start, n_pairs - rollout_buffer + 1)
remaining_pair_indices = np.arange(remaining_start, remaining_end, dtype=int)
if remaining_pair_indices.size < 2:
    raise ValueError(
        "Not enough days remain after the training split for validation and test "
        f"with rollout_buffer={rollout_buffer}. Reduce max_rollout_days or train_days."
    )

n_val = max(1, int(round(remaining_pair_indices.size * validation_fraction_of_remainder)))
n_val = min(n_val, remaining_pair_indices.size - 1)
val_pair_indices = remaining_pair_indices[:n_val]
test_pair_indices = remaining_pair_indices[n_val:]

print(f"Available daily fields: {n_days_available}")
print(f"Rollout buffer: {rollout_buffer} day(s) reserved at the tail of each split")
print(f"Training pairs: {len(train_pair_indices)} from the first {train_day_count} days")
print(f"Training order: reshuffled every epoch with seed {random_seed}")
print(f"Validation pairs: {len(val_pair_indices)}")
print(f"Test pairs: {len(test_pair_indices)}")
print(f"Training grid shape: {ssh_daily_np.shape[1:]}")

## 5. ResUNet Velocity Correction

The solver sends five channels to the network: SSH, mask, normalized `f0`, normalized `dx`, and normalized `dy`. The two output channels are velocity corrections on the original QGSW `u` and `v` supports.

In [ ]:
def normalization_groups(channels, max_groups=8):
    groups = min(max_groups, channels)
    while channels % groups != 0:
        groups -= 1
    return groups


class ResUNetBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.projection = torch.nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None
        self.block = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            torch.nn.GroupNorm(normalization_groups(out_channels), out_channels),
            torch.nn.GELU(),
            torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            torch.nn.GroupNorm(normalization_groups(out_channels), out_channels),
        )
        self.activation = torch.nn.GELU()

    def forward(self, x):
        residual = x if self.projection is None else self.projection(x)
        return self.activation(self.block(x) + residual)


class VelocityCorrectionResUNet(torch.nn.Module):
    """U-Net with residual blocks predicting (u_corr, v_corr).

    ``channels`` must be the same sequence passed as ``nn_channels`` to
    ``qgsw.qgsw``; the default ``("ssh",)`` is SSH alone, matching the ConvNet
    of arXiv:1911.08856. Taking the names rather than a bare count lets the
    module find its own "mask" channel for the input normalisation.
    """

    def __init__(self, channels=("ssh",), base_channels=16, depth=2, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.channels = tuple(channels)
        self.in_channels = len(self.channels)
        self.mask_channel_index = (
            self.channels.index("mask") if "mask" in self.channels else None
        )
        self.depth = int(depth)
        if self.depth < 1:
            raise ValueError("depth must be a positive integer (number of pooling stages)")

        # Encoder: depth + 1 ResUNetBlocks. Level k has base_channels * 2**k channels;
        # level depth is the bottleneck.
        self.encoders = torch.nn.ModuleList()
        prev_c = self.in_channels
        for level in range(self.depth + 1):
            curr_c = base_channels * (2 ** level)
            self.encoders.append(ResUNetBlock(prev_c, curr_c))
            prev_c = curr_c

        # Decoder: depth ResUNetBlocks, mirroring levels (depth-1) down to 0.
        self.decoders = torch.nn.ModuleList()
        for level in reversed(range(self.depth)):
            skip_c = base_channels * (2 ** level)
            deep_c = base_channels * (2 ** (level + 1))
            self.decoders.append(ResUNetBlock(skip_c + deep_c, skip_c))

        self.head = torch.nn.Conv2d(base_channels, 2, kernel_size=1)
        torch.nn.init.zeros_(self.head.weight)
        torch.nn.init.zeros_(self.head.bias)

    def normalize_input(self, x):
        """Standardise the SSH channel per image; pass any extra channels through.

        When a "mask" channel is supplied (nn_channels=("ssh", "mask", ...)) the
        statistics are restricted to active interior water and land is zeroed,
        so coastlines do not bias them. With SSH alone -- the default, and the
        right choice on a land-free domain like this one -- the whole patch is
        used.
        """
        if x.shape[1] != self.in_channels:
            raise ValueError(
                f"expected {self.in_channels} input channel(s), got {x.shape[1]}; "
                "in_channels must match len(nn_channels)"
            )
        ssh = x[:, 0:1]
        extra = x[:, 1:]

        if self.mask_channel_index is not None:
            m = x[:, self.mask_channel_index:self.mask_channel_index + 1]
            active = (m > 0.75).to(dtype=ssh.dtype)
        else:
            active = torch.ones_like(ssh)

        count = active.sum(dim=(-2, -1), keepdim=True).clamp_min(1.0)
        ssh_mean = (ssh * active).sum(dim=(-2, -1), keepdim=True) / count
        ssh_centered = (ssh - ssh_mean) * active
        ssh_var = (ssh_centered ** 2).sum(dim=(-2, -1), keepdim=True) / count
        ssh_std = torch.sqrt(ssh_var + self.eps)
        ssh_norm = ((ssh - ssh_mean) / ssh_std) * active

        return torch.cat([ssh_norm, extra], dim=1)

    def forward(self, x):
        x = self.normalize_input(x)

        skips = []
        feat = self.encoders[0](x)
        skips.append(feat)
        for level in range(1, self.depth):
            feat = self.encoders[level](torch.nn.functional.avg_pool2d(feat, kernel_size=2))
            skips.append(feat)
        bottleneck = self.encoders[self.depth](torch.nn.functional.avg_pool2d(feat, kernel_size=2))

        d = bottleneck
        for step, dec in enumerate(self.decoders):
            skip = skips[self.depth - 1 - step]
            up = torch.nn.functional.interpolate(d, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            d = dec(torch.cat([up, skip], dim=1))

        return self.head(d)

## 6. Train

Each epoch reshuffles the full training split with `random_seed`, then processes it in `samples_per_batch` chunks. Gradients are accumulated across the pairs in a chunk and one optimizer step is taken per chunk. The rollout horizon `k_today` follows a step-curriculum from 1 to `max_rollout_days`; the per-pair loss is the weighted mean MSE over those `k_today` daily snapshots. Validation evaluates `max_rollout_days`-day rollouts and reports the full per-day MSE curve.

In [ ]:
import QGNet_Pytorch.modgrid as modgrid_torch
import QGNet_Pytorch.modelliptic as modelliptic_torch


def finite_mse(prediction, target, valid_mask=None):
    valid = torch.isfinite(prediction) & torch.isfinite(target)
    if valid_mask is not None:
        valid = valid & valid_mask.to(device=prediction.device)
    if not torch.any(valid):
        raise ValueError("No finite points are available for the loss")
    return torch.mean((prediction[valid] - target[valid]) ** 2)


def select_pair_indices(pair_indices, max_samples=None, offset=0):
    pair_indices = np.asarray(pair_indices)
    if max_samples is None or max_samples >= len(pair_indices):
        return pair_indices
    max_samples = int(max_samples)
    if max_samples < 1:
        return np.asarray([], dtype=pair_indices.dtype)
    start = int(offset) % len(pair_indices)
    ordered = np.concatenate([pair_indices[start:], pair_indices[:start]])
    return ordered[:max_samples]


def iter_pair_batches(pair_indices, samples_per_batch):
    pair_indices = np.asarray(pair_indices, dtype=int)
    if pair_indices.size == 0:
        return
    if samples_per_batch is None or int(samples_per_batch) >= pair_indices.size:
        yield pair_indices
        return
    batch_size = int(samples_per_batch)
    for start in range(0, pair_indices.size, batch_size):
        yield pair_indices[start : start + batch_size]


def forecast_k_days_from_pair(pair_index, nn_module, *, k_days, enable_grad):
    ssh_now = ssh_daily[int(pair_index)]
    context = torch.enable_grad() if enable_grad else torch.no_grad()
    with context:
        forecast, _ = qgsw_torch(
            Hi=ssh_now,
            c=c_train,
            lon=lon_train,
            lat=lat_train,
            tint=int(k_days) * 86400,
            dtout=86400,
            dt=training_dt,
            rappel=None,
            snu=training_snu,
            nn_module=nn_module,
            nn_cadence_steps=training_nn_cadence_steps,
            nn_channels=training_nn_channels,
            # Reuse the cached grid for this sample: grid construction is
            # identical for every call with the same land mask and would
            # otherwise be repeated on every forward pass.
            grd=grid_for_pair(pair_index),
        )
    return forecast[1:]  # (k_days, ny, nx); drops the t=0 frame which equals ssh_now.


def forecast_one_day_from_pair(pair_index, nn_module, *, enable_grad):
    return forecast_k_days_from_pair(pair_index, nn_module, k_days=1, enable_grad=enable_grad)[-1]


def rollout_days_for_epoch(epoch, total_epochs, max_days):
    max_days = max(1, int(max_days))
    if max_days == 1:
        return 1
    stage_size = max(1, int(total_epochs) // max_days)
    stage = int(epoch) // stage_size
    return 1 + min(max_days - 1, stage)


def k_step_loss(pair_index, nn_module, *, k_days):
    k_days = int(k_days)
    pred_seq = forecast_k_days_from_pair(pair_index, nn_module, k_days=k_days, enable_grad=True)
    pi = int(pair_index)
    target_seq = ssh_daily[pi + 1 : pi + 1 + k_days]
    mask = loss_mask_for_pair(pi)
    ssh_losses = torch.stack([finite_mse(pred_seq[d], target_seq[d], mask) for d in range(k_days)])
    total = ssh_losses.mean()

    pv_weight = float(loss_pv_weight)
    if pv_weight > 0.0:
        pair_grid = grid_for_pair(pi)
        with torch.no_grad():
            q_target = torch.stack([modelliptic_torch.h2pv(target_seq[d], pair_grid)[0] for d in range(k_days)])
            if mask is not None:
                pv_scale = q_target[:, mask].abs().mean().clamp_min(1e-12)
            else:
                pv_scale = q_target[torch.isfinite(q_target)].abs().mean().clamp_min(1e-12)
        q_pred = torch.stack([modelliptic_torch.h2pv(pred_seq[d], pair_grid)[0] for d in range(k_days)])
        pv_losses = torch.stack([
            finite_mse(q_pred[d] / pv_scale, q_target[d] / pv_scale, mask)
            for d in range(k_days)
        ])
        total = total + pv_weight * pv_losses.mean()

    return total


def evaluate_pairs_multistep(pair_indices, nn_module, k_days, *, pv_weight=0.0):
    """Mean per-day SSH MSE across pair_indices. Optionally also returns per-day normalized PV MSE."""
    k_days = int(k_days)
    pv_weight = float(pv_weight)
    ssh_per_pair = []
    pv_per_pair = []
    for pair_index in np.asarray(pair_indices, dtype=int):
        pi = int(pair_index)
        pred_seq = forecast_k_days_from_pair(pair_index, nn_module, k_days=k_days, enable_grad=False)
        target_seq = ssh_daily[pi + 1 : pi + 1 + k_days]
        mask = loss_mask_for_pair(pi)
        ssh_per_pair.append([
            float(finite_mse(pred_seq[d], target_seq[d], mask).detach().cpu())
            for d in range(k_days)
        ])
        if pv_weight > 0.0:
            pair_grid = grid_for_pair(pi)
            q_target = torch.stack([modelliptic_torch.h2pv(target_seq[d], pair_grid)[0] for d in range(k_days)])
            if mask is not None:
                pv_scale = q_target[:, mask].abs().mean().clamp_min(1e-12)
            else:
                pv_scale = q_target[torch.isfinite(q_target)].abs().mean().clamp_min(1e-12)
            q_pred = torch.stack([modelliptic_torch.h2pv(pred_seq[d], pair_grid)[0] for d in range(k_days)])
            pv_per_pair.append([
                float(finite_mse(q_pred[d] / pv_scale, q_target[d] / pv_scale, mask).detach().cpu())
                for d in range(k_days)
            ])
    if not ssh_per_pair:
        empty = np.full(k_days, np.nan)
        return empty, (empty if pv_weight > 0.0 else None)
    ssh_arr = np.mean(ssh_per_pair, axis=0)
    pv_arr = np.mean(pv_per_pair, axis=0) if pv_per_pair else None
    return ssh_arr, pv_arr


# Pre-transfer the SSH archive to the training device once, removing per-pair host->device copies.
ssh_daily = torch.as_tensor(ssh_daily_np, dtype=training_dtype, device=training_device)
ssh_daily_mb = ssh_daily.element_size() * ssh_daily.numel() / 1024**2
print(f"ssh_daily on {ssh_daily.device}: shape={tuple(ssh_daily.shape)}, {ssh_daily_mb:.1f} MB")

c_train = torch.as_tensor(ssh_daily_c, dtype=training_dtype, device=training_device)
lon_train = torch.as_tensor(ssh_daily_lon, dtype=training_dtype, device=training_device)
lat_train = torch.as_tensor(ssh_daily_lat, dtype=training_dtype, device=training_device)

if debug_overfit_one_sample:
    if overfit_pair_offset < 0 or overfit_pair_offset >= len(train_pair_indices):
        raise ValueError("overfit_pair_offset must select an entry in train_pair_indices")
    fixed_train_pair_indices = np.asarray([train_pair_indices[int(overfit_pair_offset)]], dtype=int)
else:
    fixed_train_pair_indices = None

if samples_per_batch is not None and int(samples_per_batch) < 1:
    raise ValueError("samples_per_batch must be None or a positive integer")
if max_val_samples is not None and int(max_val_samples) < 0:
    raise ValueError("max_val_samples must be None, 0, or a positive integer")
if loss_mask_mode not in {"interior", "finite"}:
    raise ValueError("loss_mask_mode must be 'interior' or 'finite'")
if int(max_rollout_days) < 1:
    raise ValueError("max_rollout_days must be a positive integer")

pair_grid_cache = {}
loss_mask_cache = {}


def grid_for_pair(pair_index):
    pair_index = int(pair_index)
    if pair_index not in pair_grid_cache:
        with torch.no_grad():
            pair_grid_cache[pair_index] = modgrid_torch.grid(
                ssh_daily[pair_index],
                c_train,
                training_snu,
                lon_train,
                lat_train,
            )
    return pair_grid_cache[pair_index]


def loss_mask_for_pair(pair_index):
    if loss_mask_mode == "finite":
        return None
    pair_index = int(pair_index)
    if pair_index not in loss_mask_cache:
        loss_mask_cache[pair_index] = grid_for_pair(pair_index).mask == 2.0
    return loss_mask_cache[pair_index]

if fixed_train_pair_indices is not None:
    validation_pair_subset = np.asarray([], dtype=int)
else:
    validation_pair_subset = select_pair_indices(val_pair_indices, max_val_samples, offset=0)

velocity_resunet_train = VelocityCorrectionResUNet(
    channels=training_nn_channels,
    base_channels=resunet_base_channels,
    depth=resunet_depth,
).to(device=training_device, dtype=training_dtype)

for tensor_name, tensor in {
    "ssh_daily": ssh_daily,
    "c_train": c_train,
    "lon_train": lon_train,
    "lat_train": lat_train,
}.items():
    if tensor.dtype != training_dtype:
        raise TypeError(f"{tensor_name} has dtype {tensor.dtype}, expected {training_dtype}")
model_parameter_dtype = next(velocity_resunet_train.parameters()).dtype
if model_parameter_dtype != training_dtype:
    raise TypeError(f"model parameters have dtype {model_parameter_dtype}, expected {training_dtype}")

optimizer = torch.optim.AdamW(velocity_resunet_train.parameters(), lr=training_lr, weight_decay=1e-4)
training_history = []
train_pair_updates_seen = 0
optimizer_steps_seen = 0
train_shuffle_rng = np.random.default_rng(random_seed)
checkpoint_dir = TORCH_ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_weights_path = checkpoint_dir / "velocity_resunet_best.pt"
best_val_score = float("inf")
best_val_epoch = None

n_train_pairs = len(train_pair_indices)
batches_per_epoch = sum(1 for _ in iter_pair_batches(train_pair_indices, samples_per_batch))
print(f"Epochs: {num_epochs}")
print(f"Training pairs per epoch: {n_train_pairs}")
print(f"Samples per batch: {samples_per_batch}")
print(f"Optimizer steps per epoch: {batches_per_epoch}")
print(f"Fixed validation samples: {len(validation_pair_subset)} / {len(val_pair_indices)}")
print(f"Training dt={training_dt} s, lr={training_lr:g}, device={training_device}")
print(f"Model: VelocityCorrectionResUNet(in_channels={len(training_nn_channels)}, base_channels={resunet_base_channels}, depth={resunet_depth})")
print(f"CNN input channels: {training_nn_channels}")
print(f"Loss mask mode: {loss_mask_mode} ({'per-pair active interior' if loss_mask_mode == 'interior' else 'finite prediction/target points'})")
print(f"Curriculum: rollout horizon ramps from 1 to {max_rollout_days} day(s); validation reports per-day MSE up to that horizon")
print("Loss: SSH MSE" + (f" + {loss_pv_weight:g} * normalized PV MSE" if float(loss_pv_weight) > 0 else ""))
print(f"Optimizer: AdamW(lr={training_lr:g}, weight_decay=1e-4); lr_drop_factor_per_k_step={lr_drop_factor_per_k_step:g}")
if fixed_train_pair_indices is not None:
    print(f"Overfit train pair index: {int(fixed_train_pair_indices[0])}")

start_time = time.perf_counter()
prev_k_today = None
for epoch in range(int(num_epochs)):
    velocity_resunet_train.train()
    epoch_losses = []
    epoch_pairs_seen = set()
    epoch_optimizer_steps = 0

    if fixed_train_pair_indices is not None:
        epoch_train_order = fixed_train_pair_indices.copy()
        k_today = 1
    else:
        epoch_train_order = train_shuffle_rng.permutation(train_pair_indices)
        k_today = rollout_days_for_epoch(epoch, int(num_epochs), int(max_rollout_days))

    if prev_k_today is not None and k_today != prev_k_today and float(lr_drop_factor_per_k_step) != 1.0:
        for pg in optimizer.param_groups:
            pg["lr"] = pg["lr"] * float(lr_drop_factor_per_k_step)
        print(f"  [LR step] k_today {prev_k_today} -> {k_today}: lr -> {optimizer.param_groups[0]['lr']:.3e}")
    prev_k_today = k_today

    for batch_pair_indices in iter_pair_batches(epoch_train_order, samples_per_batch):
        batch_pair_indices = np.asarray(batch_pair_indices, dtype=int)
        optimizer.zero_grad(set_to_none=True)
        batch_losses = []
        batch_size = int(batch_pair_indices.size)

        for pair_index in batch_pair_indices:
            pair_index = int(pair_index)
            epoch_pairs_seen.add(pair_index)
            train_pair_updates_seen += 1
            pair_loss = k_step_loss(pair_index, velocity_resunet_train, k_days=k_today)
            (pair_loss / batch_size).backward()
            batch_losses.append(float(pair_loss.detach().cpu()))

        optimizer.step()
        optimizer_steps_seen += 1
        epoch_optimizer_steps += 1
        epoch_losses.extend(batch_losses)

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else np.nan
    velocity_resunet_train.eval()
    should_validate = max_val_samples != 0 and validation_interval is not None and (epoch + 1) % validation_interval == 0
    if should_validate:
        val_per_day, val_pv_per_day = evaluate_pairs_multistep(
            validation_pair_subset,
            velocity_resunet_train,
            int(max_rollout_days),
            pv_weight=float(loss_pv_weight),
        )
        val_loss = float(val_per_day[0]) if val_per_day.size and np.isfinite(val_per_day[0]) else np.nan
        val_ssh_mean = (
            float(np.nanmean(val_per_day))
            if val_per_day.size and np.any(np.isfinite(val_per_day))
            else np.nan
        )
        if val_pv_per_day is not None:
            val_pv_mean = (
                float(np.nanmean(val_pv_per_day))
                if val_pv_per_day.size and np.any(np.isfinite(val_pv_per_day))
                else np.nan
            )
            if np.isfinite(val_ssh_mean) and np.isfinite(val_pv_mean):
                val_score = val_ssh_mean + float(loss_pv_weight) * val_pv_mean
            else:
                val_score = np.nan
        else:
            val_pv_mean = np.nan
            val_score = val_ssh_mean
    else:
        val_per_day = np.full(int(max_rollout_days), np.nan)
        val_pv_per_day = None
        val_loss = np.nan
        val_ssh_mean = np.nan
        val_pv_mean = np.nan
        val_score = np.nan
    saved_best = False
    if np.isfinite(val_score) and val_score < best_val_score:
        best_val_score = float(val_score)
        best_val_epoch = epoch + 1
        torch.save(velocity_resunet_train.state_dict(), best_weights_path)
        saved_best = True

    training_history.append({
        "epoch": epoch + 1,
        "k_today": int(k_today),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "train_mse": train_loss,
        "val_score": val_score,
        "val_ssh_mean": val_ssh_mean,
        "val_pv_mean": val_pv_mean,
        "val_mse_day1": val_loss,
        "val_mse_per_day": val_per_day.tolist(),
        "val_pv_per_day": (val_pv_per_day.tolist() if val_pv_per_day is not None else None),
        "saved_best": saved_best,
    })
    coverage_count = len(epoch_pairs_seen)
    coverage_percent = 100.0 * coverage_count / max(n_train_pairs, 1)
    shuffle_preview = epoch_train_order[: min(5, len(epoch_train_order))].tolist()
    val_curve_str = ", ".join(
        f"d{d+1}={v:.3e}" if np.isfinite(v) else f"d{d+1}=nan"
        for d, v in enumerate(val_per_day)
    )
    print(
        f"Epoch {epoch + 1:03d}/{num_epochs} (k={k_today}): "
        f"train MSE={train_loss:.6e}, val mean MSE={val_score:.6e} | per-day [{val_curve_str}], "
        f"train pairs seen={coverage_count}/{n_train_pairs} ({coverage_percent:.1f}%), "
        f"optimizer steps={epoch_optimizer_steps}, total pair updates={train_pair_updates_seen}, "
        f"shuffle head={shuffle_preview}"
        f"{' [saved best]' if saved_best else ''}"
    )

elapsed = time.perf_counter() - start_time
print(f"Training elapsed: {elapsed:.1f} s")
if best_val_epoch is not None:
    best_state_dict = torch.load(best_weights_path, map_location=training_device)
    velocity_resunet_train.load_state_dict(best_state_dict)
    velocity_resunet_train.to(device=training_device, dtype=training_dtype)
    velocity_resunet_train.eval()
    print(f"Loaded best validation weights from epoch {best_val_epoch}: val mean MSE={best_val_score:.6e}")
    print(f"Best weights path: {best_weights_path}")
else:
    print("No finite validation loss was recorded; keeping final epoch weights.")

## 7. Loss Curves

In [ ]:
if training_history:
    epochs = np.arange(1, len(training_history) + 1)
    train_loss_array = np.asarray([h["train_mse"] for h in training_history], dtype=np.float64)
    val_score_array = np.asarray([h["val_score"] for h in training_history], dtype=np.float64)
    val_ssh_mean_array = np.asarray([h["val_ssh_mean"] for h in training_history], dtype=np.float64)
    val_pv_mean_array = np.asarray([h["val_pv_mean"] for h in training_history], dtype=np.float64)
    val_per_day_array = np.asarray([h["val_mse_per_day"] for h in training_history], dtype=np.float64)
    k_today_array = np.asarray([h["k_today"] for h in training_history], dtype=int)
    finite_score = np.isfinite(val_score_array)
    hybrid_loss_in_use = bool(np.any(np.isfinite(val_pv_mean_array)))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

    # Panel 1: train loss + val score (the selection metric)
    axes[0].plot(epochs, train_loss_array, marker="o", linewidth=1.8, label="train (mean over k_today days)")
    if finite_score.any():
        score_label = "val score (selection metric, hybrid SSH + λ·PV)" if hybrid_loss_in_use else "val score (selection metric, SSH-only mean)"
        axes[0].plot(
            epochs[finite_score],
            val_score_array[finite_score],
            marker="s",
            linewidth=1.8,
            label=score_label,
        )
    if hybrid_loss_in_use:
        finite_ssh = np.isfinite(val_ssh_mean_array)
        if finite_ssh.any():
            axes[0].plot(
                epochs[finite_ssh],
                val_ssh_mean_array[finite_ssh],
                marker=".",
                linestyle="--",
                linewidth=1.0,
                alpha=0.7,
                label="val SSH-only mean (reference)",
            )
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("MSE")
    axes[0].set_title("Forecast Loss\n(train MSE jumps at k transitions = more days enter the average)")
    axes[0].grid(True, which="both", alpha=0.3)
    axes[0].legend(fontsize=8)

    # Annotate curriculum stage transitions
    if len(k_today_array) > 1:
        boundaries = [0]
        for i in range(1, len(k_today_array)):
            if k_today_array[i] != k_today_array[i - 1]:
                boundaries.append(i)
        for idx in boundaries:
            axes[0].axvline(epochs[idx], color="grey", linestyle=":", alpha=0.4)
            axes[0].text(
                epochs[idx],
                axes[0].get_ylim()[1],
                f" k={k_today_array[idx]}",
                color="grey",
                fontsize=8,
                va="top",
            )

    # Panel 2: per-day SSH validation breakdown
    n_days = val_per_day_array.shape[1] if val_per_day_array.ndim == 2 else 0
    for d in range(n_days):
        series = val_per_day_array[:, d]
        finite_d = np.isfinite(series)
        if finite_d.any():
            axes[1].plot(epochs[finite_d], series[finite_d], marker=".", linewidth=1.4, label=f"day {d + 1}")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Validation SSH MSE")
    axes[1].set_title("Per-day validation breakdown (SSH-only)")
    axes[1].grid(True, which="both", alpha=0.3)
    if n_days > 0:
        axes[1].legend(loc="best")

    best_train_epoch = int(np.nanargmin(train_loss_array)) + 1
    print(f"Best training MSE: {np.nanmin(train_loss_array):.6e} at epoch {best_train_epoch}")
    if finite_score.any():
        best_val_index = int(np.nanargmin(val_score_array))
        print(f"Best val score: {val_score_array[best_val_index]:.6e} at epoch {best_val_index + 1} (k_today={k_today_array[best_val_index]})")
        if hybrid_loss_in_use:
            print(f"  ssh_mean={val_ssh_mean_array[best_val_index]:.6e}, pv_mean={val_pv_mean_array[best_val_index]:.6e}")
        per_day_at_best = val_per_day_array[best_val_index]
        print("  per-day SSH MSE: " + ", ".join(f"d{d + 1}={v:.3e}" if np.isfinite(v) else f"d{d + 1}=nan" for d, v in enumerate(per_day_at_best)))
    else:
        print("Validation was skipped for this run.")
    plt.show()
else:
    print("No training history available. Run the training cell first.")

## 8. One-Day Test Forecast

In [ ]:
def time_label(index):
    if "ssh_daily_time" not in globals() or ssh_daily_time is None:
        return f"index {index}"
    value = np.asarray(ssh_daily_time)[index]
    if np.issubdtype(np.asarray(value).dtype, np.datetime64):
        return np.datetime_as_string(value, unit="D")
    return str(value)


def finite_rmse(prediction, target, valid_mask=None):
    valid = torch.isfinite(prediction) & torch.isfinite(target)
    if valid_mask is not None:
        valid = valid & valid_mask.to(device=prediction.device)
    if not torch.any(valid):
        return float("nan")
    return torch.sqrt(torch.mean((prediction[valid] - target[valid]) ** 2)).item()


def robust_limits(*fields, symmetric=False, percentile=99.0):
    chunks = []
    for field in fields:
        array = np.asarray(field)
        finite = array[np.isfinite(array)].ravel()
        if finite.size:
            chunks.append(finite)
    if not chunks:
        return -1.0, 1.0
    values = np.concatenate(chunks)
    if symmetric:
        bound = np.nanpercentile(np.abs(values), percentile)
        bound = bound if np.isfinite(bound) and bound > 0 else 1.0
        return -bound, bound
    vmin, vmax = np.nanpercentile(values, [100.0 - percentile, percentile])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(values), np.nanmax(values)
    if vmin == vmax:
        vmin -= 1.0
        vmax += 1.0
    return vmin, vmax


def aggregate_per_day_rmse(pair_indices, nn_module, k_days):
    """Mean per-day SSH RMSE across pair_indices for a given model (None = QG-only)."""
    rows = []
    for pi in pair_indices:
        pi = int(pi)
        pred_seq = forecast_k_days_from_pair(pi, nn_module, k_days=k_days, enable_grad=False)
        target_seq = ssh_daily[pi + 1 : pi + 1 + k_days]
        mask = loss_mask_for_pair(pi)
        rows.append([finite_rmse(pred_seq[d], target_seq[d], mask) for d in range(k_days)])
    return np.asarray(rows, dtype=np.float64)


def aggregate_persistence_per_day_rmse(pair_indices, k_days):
    rows = []
    for pi in pair_indices:
        pi = int(pi)
        ssh_now = ssh_daily[pi]
        target_seq = ssh_daily[pi + 1 : pi + 1 + k_days]
        mask = loss_mask_for_pair(pi)
        rows.append([finite_rmse(ssh_now, target_seq[d], mask) for d in range(k_days)])
    return np.asarray(rows, dtype=np.float64)


def _print_per_day_summary(label, arr):
    means = np.nanmean(arr, axis=0)
    stds = np.nanstd(arr, axis=0)
    cells = " ".join(f"d{d + 1}={means[d]:.3e}±{stds[d]:.2e}" for d in range(arr.shape[1]))
    print(f"  {label:<13} per-day RMSE: {cells}")


if "velocity_resunet_train" not in globals() or not test_pair_indices.size:
    print("Run the training cell before this diagnostic.")
else:
    test_horizon = int(max_rollout_days)
    print(f"Aggregating test metrics over {len(test_pair_indices)} test pairs at horizon {test_horizon} day(s)...")
    velocity_resunet_train.eval()
    hybrid_rmse_arr = aggregate_per_day_rmse(test_pair_indices, velocity_resunet_train, test_horizon)
    qg_only_rmse_arr = aggregate_per_day_rmse(test_pair_indices, None, test_horizon)
    persistence_rmse_arr = aggregate_persistence_per_day_rmse(test_pair_indices, test_horizon)
    _print_per_day_summary("ResUNet-QGSW", hybrid_rmse_arr)
    _print_per_day_summary("QG-only", qg_only_rmse_arr)
    _print_per_day_summary("Persistence", persistence_rmse_arr)
    print()
    test_pair_index = int(test_pair_indices[0])
    test_loss_mask = loss_mask_for_pair(test_pair_index) if "loss_mask_for_pair" in globals() else None
    velocity_resunet_train.eval()
    with torch.no_grad():
        test_input = ssh_daily[test_pair_index]
        test_target = ssh_daily[test_pair_index + 1]
        test_pred = forecast_one_day_from_pair(test_pair_index, velocity_resunet_train, enable_grad=False)
        qg_only_pred = forecast_one_day_from_pair(test_pair_index, None, enable_grad=False)
        persistence_pred = test_input

    hybrid_rmse = finite_rmse(test_pred, test_target, test_loss_mask)
    qg_only_rmse = finite_rmse(qg_only_pred, test_target, test_loss_mask)
    persistence_rmse = finite_rmse(persistence_pred, test_target, test_loss_mask)

    input_np = test_input.detach().cpu().numpy()
    target_np = test_target.detach().cpu().numpy()
    pred_np = test_pred.detach().cpu().numpy()
    qg_only_np = qg_only_pred.detach().cpu().numpy()
    persistence_np = persistence_pred.detach().cpu().numpy()
    hybrid_error = pred_np - target_np
    qg_only_error = qg_only_np - target_np
    persistence_error = persistence_np - target_np
    error_reduction = np.abs(qg_only_error) - np.abs(hybrid_error)

    state_vmin, state_vmax = robust_limits(input_np, target_np, pred_np, qg_only_np, persistence_np)
    err_vmin, err_vmax = robust_limits(hybrid_error, qg_only_error, persistence_error, symmetric=True)
    improvement_vmin, improvement_vmax = robust_limits(error_reduction, symmetric=True)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
    state_specs = [
        (axes[0, 0], input_np, "Input SSH(t)"),
        (axes[0, 1], target_np, "Target SSH(t+1 day)"),
        (axes[0, 2], qg_only_np, f"QG-only forecast\nRMSE={qg_only_rmse:.4e}"),
        (axes[0, 3], pred_np, f"ResUNet-QGSW forecast\nRMSE={hybrid_rmse:.4e}"),
    ]
    error_specs = [
        (axes[1, 0], persistence_error, f"Persistence error\nRMSE={persistence_rmse:.4e}"),
        (axes[1, 1], qg_only_error, f"QG-only error\nRMSE={qg_only_rmse:.4e}"),
        (axes[1, 2], hybrid_error, f"ResUNet-QGSW error\nRMSE={hybrid_rmse:.4e}"),
    ]

    for ax, field, title in state_specs:
        im_state = ax.imshow(np.ma.masked_invalid(field), origin="lower", cmap="viridis", vmin=state_vmin, vmax=state_vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax, field, title in error_specs:
        im_error = ax.imshow(np.ma.masked_invalid(field), origin="lower", cmap="seismic", vmin=err_vmin, vmax=err_vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    im_improvement = axes[1, 3].imshow(
        np.ma.masked_invalid(error_reduction),
        origin="lower",
        cmap="BrBG",
        vmin=improvement_vmin,
        vmax=improvement_vmax,
    )
    axes[1, 3].set_title("Error reduction\npositive=ResUNet better")
    axes[1, 3].set_xticks([])
    axes[1, 3].set_yticks([])

    fig.colorbar(im_state, ax=axes[0, :], fraction=0.03, pad=0.02, label="SSH (m)")
    fig.colorbar(im_error, ax=axes[1, :3], fraction=0.03, pad=0.02, label="SSH error (m)")
    fig.colorbar(im_improvement, ax=axes[1, 3], fraction=0.05, pad=0.02, label="|QG error| - |ResUNet error| (m)")
    fig.suptitle(f"One-day test forecast: pair {test_pair_index} ({time_label(test_pair_index)} to {time_label(test_pair_index + 1)})")

    print(f"ResUNet-QGSW SSH RMSE: {hybrid_rmse:.6e}")
    print(f"QG-only SSH RMSE:      {qg_only_rmse:.6e}")
    print(f"Persistence SSH RMSE:  {persistence_rmse:.6e}")
    print(f"ResUNet minus QG-only RMSE: {hybrid_rmse - qg_only_rmse:+.6e}")
    plt.show()


## 9. Potential Vorticity Diagnostic

This cell converts SSH fields to QGSW potential vorticity and compares the one-day ResUNet-QGSW forecast with the target, QG-only forecast, and persistence baseline.


In [ ]:
import QGNet_Pytorch.modelliptic as modelliptic_torch


if "time_label" not in globals():
    def time_label(index):
        if "ssh_daily_time" not in globals() or ssh_daily_time is None:
            return f"index {index}"
        value = np.asarray(ssh_daily_time)[index]
        if np.issubdtype(np.asarray(value).dtype, np.datetime64):
            return np.datetime_as_string(value, unit="D")
        return str(value)


def pv_grid_for(ssh_tensor):
    return modgrid_torch.grid(
        ssh_tensor,
        c_train,
        training_snu,
        lon_train,
        lat_train,
    )


def potential_vorticity(ssh_tensor):
    grid = pv_grid_for(ssh_tensor)
    q, = modelliptic_torch.h2pv(ssh_tensor, grid)
    q = torch.where(grid.mask == 2.0, q, torch.full_like(q, torch.nan))
    return q


def finite_torch_rmse(prediction, target, valid_mask=None):
    valid = torch.isfinite(prediction) & torch.isfinite(target)
    if valid_mask is not None:
        valid = valid & valid_mask.to(device=prediction.device)
    if not torch.any(valid):
        return float("nan")
    return torch.sqrt(torch.mean((prediction[valid] - target[valid]) ** 2)).item()


def finite_limits_np(*fields, symmetric=False, percentile=99.0):
    chunks = []
    for field in fields:
        array = np.asarray(field)
        finite = array[np.isfinite(array)].ravel()
        if finite.size:
            chunks.append(finite)
    if not chunks:
        return -1.0, 1.0
    values = np.concatenate(chunks)
    if symmetric:
        bound = np.nanpercentile(np.abs(values), percentile)
        if not np.isfinite(bound) or bound <= 0:
            bound = np.nanmax(np.abs(values)) if values.size else 1.0
        if not np.isfinite(bound) or bound <= 0:
            bound = 1.0
        return -bound, bound
    vmin, vmax = np.nanpercentile(values, [100.0 - percentile, percentile])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = np.nanmin(values), np.nanmax(values)
    if vmin == vmax:
        vmin -= 1.0
        vmax += 1.0
    return vmin, vmax


if "velocity_resunet_train" not in globals() or not test_pair_indices.size:
    print("Run the training cell before this diagnostic.")
else:
    pv_test_pair_index = int(test_pair_indices[0])
    pv_loss_mask = loss_mask_for_pair(pv_test_pair_index) if "loss_mask_for_pair" in globals() else None
    velocity_resunet_train.eval()
    with torch.no_grad():
        pv_input_ssh = ssh_daily[pv_test_pair_index]
        pv_target_ssh = ssh_daily[pv_test_pair_index + 1]
        pv_forecast_ssh = forecast_one_day_from_pair(pv_test_pair_index, velocity_resunet_train, enable_grad=False)
        pv_qg_only_ssh = forecast_one_day_from_pair(pv_test_pair_index, None, enable_grad=False)
        pv_persistence_ssh = pv_input_ssh

        pv_input = potential_vorticity(pv_input_ssh)
        pv_target = potential_vorticity(pv_target_ssh)
        pv_forecast = potential_vorticity(pv_forecast_ssh)
        pv_qg_only = potential_vorticity(pv_qg_only_ssh)
        pv_persistence = potential_vorticity(pv_persistence_ssh)

    forecast_pv_rmse = finite_torch_rmse(pv_forecast, pv_target, pv_loss_mask)
    qg_only_pv_rmse = finite_torch_rmse(pv_qg_only, pv_target, pv_loss_mask)
    persistence_pv_rmse = finite_torch_rmse(pv_persistence, pv_target, pv_loss_mask)

    pv_input_np = pv_input.detach().cpu().numpy()
    pv_target_np = pv_target.detach().cpu().numpy()
    pv_forecast_np = pv_forecast.detach().cpu().numpy()
    pv_qg_only_np = pv_qg_only.detach().cpu().numpy()
    pv_persistence_np = pv_persistence.detach().cpu().numpy()
    forecast_pv_error = pv_forecast_np - pv_target_np
    qg_only_pv_error = pv_qg_only_np - pv_target_np
    persistence_pv_error = pv_persistence_np - pv_target_np
    pv_error_reduction = np.abs(qg_only_pv_error) - np.abs(forecast_pv_error)

    pv_vmin, pv_vmax = finite_limits_np(pv_input_np, pv_target_np, pv_forecast_np, pv_qg_only_np, pv_persistence_np, symmetric=True)
    err_vmin, err_vmax = finite_limits_np(forecast_pv_error, qg_only_pv_error, persistence_pv_error, symmetric=True)
    improvement_vmin, improvement_vmax = finite_limits_np(pv_error_reduction, symmetric=True)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
    pv_specs = [
        (axes[0, 0], pv_input_np, "PV from SSH(t)"),
        (axes[0, 1], pv_target_np, "Target PV(t+1 day)"),
        (axes[0, 2], pv_qg_only_np, f"QG-only PV\nRMSE={qg_only_pv_rmse:.4e}"),
        (axes[0, 3], pv_forecast_np, f"ResUNet-QGSW PV\nRMSE={forecast_pv_rmse:.4e}"),
    ]
    error_specs = [
        (axes[1, 0], persistence_pv_error, f"Persistence PV error\nRMSE={persistence_pv_rmse:.4e}"),
        (axes[1, 1], qg_only_pv_error, f"QG-only PV error\nRMSE={qg_only_pv_rmse:.4e}"),
        (axes[1, 2], forecast_pv_error, f"ResUNet-QGSW PV error\nRMSE={forecast_pv_rmse:.4e}"),
    ]

    for ax, field, title in pv_specs:
        im_pv = ax.imshow(np.ma.masked_invalid(field), origin="lower", cmap="RdBu_r", vmin=pv_vmin, vmax=pv_vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax, field, title in error_specs:
        im_err = ax.imshow(np.ma.masked_invalid(field), origin="lower", cmap="seismic", vmin=err_vmin, vmax=err_vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    im_improvement = axes[1, 3].imshow(
        np.ma.masked_invalid(pv_error_reduction),
        origin="lower",
        cmap="BrBG",
        vmin=improvement_vmin,
        vmax=improvement_vmax,
    )
    axes[1, 3].set_title("PV error reduction\npositive=ResUNet better")
    axes[1, 3].set_xticks([])
    axes[1, 3].set_yticks([])

    fig.colorbar(im_pv, ax=axes[0, :], fraction=0.03, pad=0.02, label="Potential vorticity")
    fig.colorbar(im_err, ax=axes[1, :3], fraction=0.03, pad=0.02, label="PV error")
    fig.colorbar(im_improvement, ax=axes[1, 3], fraction=0.05, pad=0.02, label="|QG PV error| - |ResUNet PV error|")
    fig.suptitle(
        f"One-day PV diagnostic: pair {pv_test_pair_index} "
        f"({time_label(pv_test_pair_index)} to {time_label(pv_test_pair_index + 1)})"
    )

    print(f"ResUNet-QGSW PV RMSE: {forecast_pv_rmse:.6e}")
    print(f"QG-only PV RMSE:      {qg_only_pv_rmse:.6e}")
    print(f"Persistence PV RMSE:  {persistence_pv_rmse:.6e}")
    print(f"ResUNet minus QG-only PV RMSE: {forecast_pv_rmse - qg_only_pv_rmse:+.6e}")
    plt.show()


## 10. Potential Vorticity and Neural-Correction Animation

This optional final cell runs a short autoregressive forecast on the test split and animates potential vorticity for the target sequence, the ResUNet-corrected model, the QG-only model, PV errors, PV error reduction, and the two velocity-correction channels produced by the neural network.


In [ ]:
pv_animation_days = 5
selected_pv_animation_test_pair = None  # Set to an integer test pair index to override test_pair_indices[0].
pv_animation_dt = training_dt
pv_animation_cadence_seconds = nn_cadence_seconds

if "modelliptic_torch" not in globals():
    import QGNet_Pytorch.modelliptic as modelliptic_torch

if "time_label" not in globals():
    def time_label(index):
        if "ssh_daily_time" not in globals() or ssh_daily_time is None:
            return f"index {index}"
        value = np.asarray(ssh_daily_time)[index]
        if np.issubdtype(np.asarray(value).dtype, np.datetime64):
            return np.datetime_as_string(value, unit="D")
        return str(value)

if "potential_vorticity" not in globals():
    def potential_vorticity(ssh_tensor):
        grid = modgrid_torch.grid(ssh_tensor, c_train, training_snu, lon_train, lat_train)
        q, = modelliptic_torch.h2pv(ssh_tensor, grid)
        q = torch.where(grid.mask == 2.0, q, torch.full_like(q, torch.nan))
        return q

if "finite_limits_np" not in globals():
    def finite_limits_np(*fields, symmetric=False, percentile=99.0):
        chunks = []
        for field in fields:
            array = np.asarray(field)
            finite = array[np.isfinite(array)].ravel()
            if finite.size:
                chunks.append(finite)
        if not chunks:
            return -1.0, 1.0
        values = np.concatenate(chunks)
        if symmetric:
            bound = np.nanpercentile(np.abs(values), percentile)
            if not np.isfinite(bound) or bound <= 0:
                bound = np.nanmax(np.abs(values)) if values.size else 1.0
            if not np.isfinite(bound) or bound <= 0:
                bound = 1.0
            return -bound, bound
        vmin, vmax = np.nanpercentile(values, [100.0 - percentile, percentile])
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
            vmin, vmax = np.nanmin(values), np.nanmax(values)
        if vmin == vmax:
            vmin -= 1.0
            vmax += 1.0
        return vmin, vmax


def normalized_grid_channel(values, active, eps=1e-12):
    values = torch.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    valid = active & torch.isfinite(values)
    if not bool(valid.any().detach().cpu()):
        return torch.zeros_like(values)
    selected = values[valid]
    std = selected.std(unbiased=False).clamp_min(eps)
    return torch.where(active, (values - selected.mean()) / std, torch.zeros_like(values))


def neural_input_for(ssh_tensor, grid):
    active = grid.mask == 2.0
    return torch.stack(
        (
            torch.nan_to_num(ssh_tensor, nan=0.0),
            grid.mask / 2.0,
            normalized_grid_channel(grid.f0, active),
            normalized_grid_channel(grid.dx, active),
            normalized_grid_channel(grid.dy, active),
        ),
        dim=0,
    ).unsqueeze(0)


def neural_u_support(grid):
    valid = torch.zeros((grid.ny, grid.nx), dtype=torch.bool, device=grid.mask.device)
    if grid.ny > 4 and grid.nx > 4:
        valid[2:-2, 2:-1] = grid.mask[2:-2, 2:-1] == 2.0
    return valid


def neural_v_support(grid):
    valid = torch.zeros((grid.ny, grid.nx), dtype=torch.bool, device=grid.mask.device)
    if grid.ny > 4 and grid.nx > 4:
        valid[2:-1, 2:-2] = grid.mask[2:-1, 2:-2] == 2.0
    return valid


def neural_velocity_outputs(ssh_tensor):
    grid = modgrid_torch.grid(ssh_tensor, c_train, training_snu, lon_train, lat_train)
    correction = velocity_resunet_train(neural_input_for(ssh_tensor, grid))
    if correction.shape != (1, 2, grid.ny, grid.nx):
        raise ValueError(f"Expected neural correction shape {(1, 2, grid.ny, grid.nx)}, got {tuple(correction.shape)}")
    u_corr = correction[0, 0]
    v_corr = correction[0, 1]
    u_corr = torch.where(neural_u_support(grid), u_corr, torch.full_like(u_corr, torch.nan))
    v_corr = torch.where(neural_v_support(grid), v_corr, torch.full_like(v_corr, torch.nan))
    return u_corr, v_corr

if "velocity_resunet_train" not in globals() or not test_pair_indices.size:
    print("Run the training cell before this animation.")
elif "ssh_daily" not in globals():
    print("Run the training cell first so ssh_daily exists.")
else:
    from IPython.display import HTML, display
    from matplotlib import animation

    pv_animation_test_pair = int(test_pair_indices[0] if selected_pv_animation_test_pair is None else selected_pv_animation_test_pair)
    pv_animation_cadence_steps = cadence_steps_for_dt(pv_animation_dt, pv_animation_cadence_seconds)

    if pv_animation_test_pair + pv_animation_days >= int(ssh_daily.shape[0]):
        print(
            "The selected test snapshot is too close to the end of ssh_daily for a "
            f"{pv_animation_days}-day ground-truth comparison. Choose an earlier selected_pv_animation_test_pair."
        )
    else:
        def run_one_day(current_ssh, nn_module):
            forecast, _ = qgsw_torch(
                Hi=current_ssh,
                c=c_train,
                lon=lon_train,
                lat=lat_train,
                tint=86400,
                dtout=86400,
                dt=pv_animation_dt,
                rappel=None,
                snu=training_snu,
                nn_module=nn_module,
            nn_channels=training_nn_channels,
                nn_cadence_steps=pv_animation_cadence_steps,
            )
            return forecast[-1].detach()

        velocity_resunet_train.eval()
        truth_states = [
            ssh_daily[pv_animation_test_pair + offset].detach().clone()
            for offset in range(pv_animation_days + 1)
        ]
        resunet_states = [truth_states[0].detach().clone()]
        qg_only_states = [truth_states[0].detach().clone()]

        with torch.no_grad():
            resunet_current = truth_states[0].detach().clone()
            qg_only_current = truth_states[0].detach().clone()
            for _ in range(pv_animation_days):
                resunet_current = run_one_day(resunet_current, velocity_resunet_train)
                qg_only_current = run_one_day(qg_only_current, None)
                resunet_states.append(resunet_current.clone())
                qg_only_states.append(qg_only_current.clone())

            truth_pv = [potential_vorticity(state).detach().cpu().numpy() for state in truth_states]
            resunet_pv = [potential_vorticity(state).detach().cpu().numpy() for state in resunet_states]
            qg_only_pv = [potential_vorticity(state).detach().cpu().numpy() for state in qg_only_states]
            neural_u_corr = []
            neural_v_corr = []
            for state in resunet_states:
                u_corr, v_corr = neural_velocity_outputs(state)
                neural_u_corr.append(u_corr.detach().cpu().numpy())
                neural_v_corr.append(v_corr.detach().cpu().numpy())

        resunet_error = [resunet_pv[i] - truth_pv[i] for i in range(len(truth_pv))]
        qg_only_error = [qg_only_pv[i] - truth_pv[i] for i in range(len(truth_pv))]
        error_reduction = [np.abs(qg_only_error[i]) - np.abs(resunet_error[i]) for i in range(len(truth_pv))]

        pv_limits = finite_limits_np(*truth_pv, *resunet_pv, *qg_only_pv, symmetric=True)
        error_limits = finite_limits_np(*resunet_error, *qg_only_error, symmetric=True)
        reduction_limits = finite_limits_np(*error_reduction, symmetric=True)
        correction_limits = finite_limits_np(*neural_u_corr, *neural_v_corr, symmetric=True)
        frame_labels = [time_label(pv_animation_test_pair + offset) for offset in range(pv_animation_days + 1)]

        fig, axes = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=False)
        fig.subplots_adjust(left=0.035, right=0.855, top=0.88, bottom=0.07, wspace=0.08, hspace=0.25)
        ax_truth, ax_resunet, ax_qg_only, ax_res_error = axes[0]
        ax_qg_error, ax_reduction, ax_u_corr, ax_v_corr = axes[1]

        imshow_kwargs = {"origin": "lower", "animated": True, "interpolation": "nearest"}
        im_truth = ax_truth.imshow(np.ma.masked_invalid(truth_pv[0]), cmap="RdBu_r", vmin=pv_limits[0], vmax=pv_limits[1], **imshow_kwargs)
        im_resunet = ax_resunet.imshow(np.ma.masked_invalid(resunet_pv[0]), cmap="RdBu_r", vmin=pv_limits[0], vmax=pv_limits[1], **imshow_kwargs)
        im_qg_only = ax_qg_only.imshow(np.ma.masked_invalid(qg_only_pv[0]), cmap="RdBu_r", vmin=pv_limits[0], vmax=pv_limits[1], **imshow_kwargs)
        im_res_error = ax_res_error.imshow(np.ma.masked_invalid(resunet_error[0]), cmap="seismic", vmin=error_limits[0], vmax=error_limits[1], **imshow_kwargs)
        im_qg_error = ax_qg_error.imshow(np.ma.masked_invalid(qg_only_error[0]), cmap="seismic", vmin=error_limits[0], vmax=error_limits[1], **imshow_kwargs)
        im_reduction = ax_reduction.imshow(np.ma.masked_invalid(error_reduction[0]), cmap="BrBG", vmin=reduction_limits[0], vmax=reduction_limits[1], **imshow_kwargs)
        im_u_corr = ax_u_corr.imshow(np.ma.masked_invalid(neural_u_corr[0]), cmap="PuOr", vmin=correction_limits[0], vmax=correction_limits[1], **imshow_kwargs)
        im_v_corr = ax_v_corr.imshow(np.ma.masked_invalid(neural_v_corr[0]), cmap="PuOr", vmin=correction_limits[0], vmax=correction_limits[1], **imshow_kwargs)

        panel_titles = {
            ax_truth: "Target PV",
            ax_resunet: "ResUNet-QGSW PV",
            ax_qg_only: "QG-only PV",
            ax_res_error: "ResUNet PV error",
            ax_qg_error: "QG-only PV error",
            ax_reduction: "PV error reduction\npositive=ResUNet better",
            ax_u_corr: "Neural output: u correction",
            ax_v_corr: "Neural output: v correction",
        }
        for ax, title in panel_titles.items():
            ax.set_title(title, fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])

        colorbar_specs = [
            (im_truth, [0.875, 0.675, 0.014, 0.18], "Potential vorticity"),
            (im_res_error, [0.875, 0.455, 0.014, 0.18], "PV error"),
            (im_reduction, [0.875, 0.245, 0.014, 0.18], "PV error reduction"),
            (im_u_corr, [0.875, 0.075, 0.014, 0.14], "Velocity correction"),
        ]
        for image, cax_bounds, label in colorbar_specs:
            cax = fig.add_axes(cax_bounds)
            cbar = fig.colorbar(image, cax=cax)
            cbar.ax.tick_params(labelsize=8)
            cbar.set_label(label, fontsize=9, labelpad=8)

        suptitle = fig.suptitle("", y=0.965, fontsize=12)

        def update(frame_index):
            im_truth.set_data(np.ma.masked_invalid(truth_pv[frame_index]))
            im_resunet.set_data(np.ma.masked_invalid(resunet_pv[frame_index]))
            im_qg_only.set_data(np.ma.masked_invalid(qg_only_pv[frame_index]))
            im_res_error.set_data(np.ma.masked_invalid(resunet_error[frame_index]))
            im_qg_error.set_data(np.ma.masked_invalid(qg_only_error[frame_index]))
            im_reduction.set_data(np.ma.masked_invalid(error_reduction[frame_index]))
            im_u_corr.set_data(np.ma.masked_invalid(neural_u_corr[frame_index]))
            im_v_corr.set_data(np.ma.masked_invalid(neural_v_corr[frame_index]))
            suptitle.set_text(
                f"{pv_animation_days}-day PV free run from test index {pv_animation_test_pair} | "
                f"day {frame_index} | {frame_labels[frame_index]}"
            )
            return [im_truth, im_resunet, im_qg_only, im_res_error, im_qg_error, im_reduction, im_u_corr, im_v_corr, suptitle]

        pv_animation_obj = animation.FuncAnimation(
            fig,
            update,
            frames=len(truth_pv),
            interval=1000,
            blit=False,
            repeat=True,
        )

        print(f"PV animation days: {pv_animation_days}")
        print(f"PV animation dt: {pv_animation_dt} s")
        print(f"Correction cadence: every {pv_animation_cadence_seconds} s ({pv_animation_cadence_steps} model steps)")
        print("Neural outputs are evaluated on each ResUNet free-run state and masked to the solver's u/v supports.")
        display(HTML(pv_animation_obj.to_jshtml()))
        plt.close(fig)


In [ ]:
gif_output_path = TORCH_ROOT / "notebooks" / "pv_animation.gif"
gif_fps = 1  # one frame per simulated day, matching interval=1000 ms in cell 20

required_globals = (
    "truth_pv", "resunet_pv", "qg_only_pv",
    "resunet_error", "qg_only_error", "error_reduction",
    "neural_u_corr", "neural_v_corr", "frame_labels",
    "pv_limits", "error_limits", "reduction_limits", "correction_limits",
    "pv_animation_days", "pv_animation_test_pair",
)
missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    print(f"Run the previous cell first; missing: {missing_globals}")
else:
    from matplotlib import animation

    fig_gif, axes_gif = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=False)
    fig_gif.subplots_adjust(left=0.035, right=0.855, top=0.88, bottom=0.07, wspace=0.08, hspace=0.25)
    (ax_truth_g, ax_resunet_g, ax_qg_only_g, ax_res_error_g) = axes_gif[0]
    (ax_qg_error_g, ax_reduction_g, ax_u_corr_g, ax_v_corr_g) = axes_gif[1]

    imshow_kwargs = {"origin": "lower", "animated": True, "interpolation": "nearest"}
    im_truth_g = ax_truth_g.imshow(np.ma.masked_invalid(truth_pv[0]), cmap="RdBu_r", vmin=pv_limits[0], vmax=pv_limits[1], **imshow_kwargs)
    im_resunet_g = ax_resunet_g.imshow(np.ma.masked_invalid(resunet_pv[0]), cmap="RdBu_r", vmin=pv_limits[0], vmax=pv_limits[1], **imshow_kwargs)
    im_qg_only_g = ax_qg_only_g.imshow(np.ma.masked_invalid(qg_only_pv[0]), cmap="RdBu_r", vmin=pv_limits[0], vmax=pv_limits[1], **imshow_kwargs)
    im_res_error_g = ax_res_error_g.imshow(np.ma.masked_invalid(resunet_error[0]), cmap="seismic", vmin=error_limits[0], vmax=error_limits[1], **imshow_kwargs)
    im_qg_error_g = ax_qg_error_g.imshow(np.ma.masked_invalid(qg_only_error[0]), cmap="seismic", vmin=error_limits[0], vmax=error_limits[1], **imshow_kwargs)
    im_reduction_g = ax_reduction_g.imshow(np.ma.masked_invalid(error_reduction[0]), cmap="BrBG", vmin=reduction_limits[0], vmax=reduction_limits[1], **imshow_kwargs)
    im_u_corr_g = ax_u_corr_g.imshow(np.ma.masked_invalid(neural_u_corr[0]), cmap="PuOr", vmin=correction_limits[0], vmax=correction_limits[1], **imshow_kwargs)
    im_v_corr_g = ax_v_corr_g.imshow(np.ma.masked_invalid(neural_v_corr[0]), cmap="PuOr", vmin=correction_limits[0], vmax=correction_limits[1], **imshow_kwargs)

    panel_titles_gif = {
        ax_truth_g: "Target PV",
        ax_resunet_g: "ResUNet-QGSW PV",
        ax_qg_only_g: "QG-only PV",
        ax_res_error_g: "ResUNet PV error",
        ax_qg_error_g: "QG-only PV error",
        ax_reduction_g: "PV error reduction\npositive=ResUNet better",
        ax_u_corr_g: "Neural output: u correction",
        ax_v_corr_g: "Neural output: v correction",
    }
    for ax, title in panel_titles_gif.items():
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    colorbar_specs_gif = [
        (im_truth_g, [0.875, 0.675, 0.014, 0.18], "Potential vorticity"),
        (im_res_error_g, [0.875, 0.455, 0.014, 0.18], "PV error"),
        (im_reduction_g, [0.875, 0.245, 0.014, 0.18], "PV error reduction"),
        (im_u_corr_g, [0.875, 0.075, 0.014, 0.14], "Velocity correction"),
    ]
    for image, cax_bounds, label in colorbar_specs_gif:
        cax = fig_gif.add_axes(cax_bounds)
        cbar = fig_gif.colorbar(image, cax=cax)
        cbar.ax.tick_params(labelsize=8)
        cbar.set_label(label, fontsize=9, labelpad=8)

    suptitle_gif = fig_gif.suptitle("", y=0.965, fontsize=12)

    def update_gif(frame_index):
        im_truth_g.set_data(np.ma.masked_invalid(truth_pv[frame_index]))
        im_resunet_g.set_data(np.ma.masked_invalid(resunet_pv[frame_index]))
        im_qg_only_g.set_data(np.ma.masked_invalid(qg_only_pv[frame_index]))
        im_res_error_g.set_data(np.ma.masked_invalid(resunet_error[frame_index]))
        im_qg_error_g.set_data(np.ma.masked_invalid(qg_only_error[frame_index]))
        im_reduction_g.set_data(np.ma.masked_invalid(error_reduction[frame_index]))
        im_u_corr_g.set_data(np.ma.masked_invalid(neural_u_corr[frame_index]))
        im_v_corr_g.set_data(np.ma.masked_invalid(neural_v_corr[frame_index]))
        suptitle_gif.set_text(
            f"{pv_animation_days}-day PV free run from test index {pv_animation_test_pair} | "
            f"day {frame_index} | {frame_labels[frame_index]}"
        )
        return [im_truth_g, im_resunet_g, im_qg_only_g, im_res_error_g, im_qg_error_g, im_reduction_g, im_u_corr_g, im_v_corr_g, suptitle_gif]

    pv_animation_gif_obj = animation.FuncAnimation(
        fig_gif,
        update_gif,
        frames=len(truth_pv),
        interval=1000,
        blit=False,
        repeat=True,
    )

    gif_output_path.parent.mkdir(parents=True, exist_ok=True)
    pv_animation_gif_obj.save(gif_output_path, writer=animation.PillowWriter(fps=gif_fps))
    plt.close(fig_gif)

    gif_size_mb = gif_output_path.stat().st_size / 1024**2
    print(f"GIF saved to {gif_output_path} ({gif_size_mb:.2f} MB, {len(truth_pv)} frames @ {gif_fps} fps)")
